# Driver Drowsiness Detection
EAR + CNN eye state + temporal fusion. Runs end to end in Colab: setup, data, training, evaluation, video demo.

**Runtime:** Runtime > Change runtime type > GPU (optional, the CNN is small).

## 1. Setup
Upload `Driver_Drowsiness_Detection.zip` (or clone your GitHub repo) and install requirements.

In [ ]:
from google.colab import files
import os, zipfile
if not os.path.exists('Driver_Drowsiness_Detection'):
    up = files.upload()                       # choose Driver_Drowsiness_Detection.zip
    zipfile.ZipFile(list(up)[0]).extractall('.')
%cd Driver_Drowsiness_Detection
!pip install -q -r requirements.txt

## 2. Dataset
Put eye images in `data/eyes/closed` and `data/eyes/open`.

**MRL Eye Dataset:** download it (for example from Kaggle), unzip to `/content/mrl`, then sort it with the helper script (uses the eye-state field in each filename).

In [ ]:
# Option A: MRL Eye Dataset already unzipped at /content/mrl
!python scripts/prepare_mrl.py /content/mrl --limit 15000

In [ ]:
import glob
for c in ('closed', 'open'):
    print(c, len(glob.glob(f'data/eyes/{c}/*')))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg, random, glob
fig, axs = plt.subplots(2, 6, figsize=(12, 4.5))
for r, c in enumerate(('closed', 'open')):
    for ax, p in zip(axs[r], random.sample(glob.glob(f'data/eyes/{c}/*'), 6)):
        ax.imshow(mpimg.imread(p), cmap='gray'); ax.set_title(c); ax.axis('off')
plt.tight_layout()

## 3. Train the eye-state CNN (70/15/15 stratified split)
Saves `models/eye_state_cnn.keras`, `outputs/confusion_matrix.png` and `outputs/training_curves.png`.

In [ ]:
!python -m src.eye_cnn --data data/eyes --epochs 20

In [ ]:
from IPython.display import Image, display
display(Image('outputs/training_curves.png')); display(Image('outputs/confusion_matrix.png'))

## 4. Run on a video
Colab has no webcam, so upload a short clip of a face (eyes closing for a few seconds works well).

In [ ]:
from google.colab import files
up = files.upload(); video = list(up)[0]
!python -m src.video_pipeline --video "{video}" --out outputs/annotated.mp4 --csv outputs/log.csv

In [ ]:
# re-encode to H.264 so the browser can play it
!ffmpeg -y -loglevel error -i outputs/annotated.mp4 -vcodec libx264 -pix_fmt yuv420p outputs/annotated_h264.mp4
from IPython.display import Video
Video('outputs/annotated_h264.mp4', embed=True, width=640)

In [ ]:
import pandas as pd, plotly.graph_objects as go
df = pd.read_csv('outputs/log.csv')
fig = go.Figure(go.Scatter(x=df.t, y=df.ear_smooth, name='EAR'))
fig.add_hline(y=0.21, line_dash='dash'); fig.update_layout(title='EAR over time', xaxis_title='s')
fig.show()
df.state.value_counts()

## 5. Compare EAR-only vs fused
Run the same clip with `--no-cnn` and compare the `state` columns of the two logs (accuracy, false alarms, detection delay).

In [ ]:
!python -m src.video_pipeline --video "{video}" --out outputs/ear_only.mp4 --csv outputs/log_ear_only.csv --no-cnn
a, b = pd.read_csv('outputs/log_ear_only.csv'), pd.read_csv('outputs/log.csv')
print('EAR only  SLEEPING frames:', (a.state == 'SLEEPING').sum())
print('Fused     SLEEPING frames:', (b.state == 'SLEEPING').sum())

## 6. Download results
Download `models/eye_state_cnn.keras` and commit it to your repo so Streamlit Cloud can use it.

In [ ]:
from google.colab import files
files.download('models/eye_state_cnn.keras')